# 1. Import and Hardware Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset
from torchvision.transforms import autoaugment

!pip install tqdm ipywidgets -q
from tqdm.auto import tqdm

import math


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
DATA_PATH = '.Data'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


# 2. Hyperparameters

1. Normal parameters

In [26]:
IN_CHANNELS = 3
BATCH_SIZE = 128
NUM_CLASSES = 101

MODEL_NAME = "EfficientNetV2-S"
EPOCHS = 150
LR = 1e-3
SEED = 42
DROPOUT = 0

2. Adaptive regularization

In [ ]:
progressive_configs = {
    "EfficientNetV2-S": {
        "img_size": (128, 300),
        "randaug_mag": (5, 15),
        "mixup_alpha": (0, 0),
        "dropout": (0.1, 0.3),
    },
    "EfficientNetV2-M": {
        "img_size": (128, 380),
        "randaug_mag": (5, 20),
        "mixup_alpha": (0, 0.2),
        "dropout": (0.1, 0.4),
    },
    "EfficientNetV2-L": {
        "img_size": (128, 380),
        "randaug_mag": (5, 25),
        "mixup_alpha": (0, 0.4),
        "dropout": (0.1, 0.5),
    },
    "EfficientNetV2-XL": {
        "img_size": (128, 380),
        "randaug_mag": (5, 25),
        "mixup_alpha": (0, 0.4),
        "dropout": (0.1, 0.5),
    },
}

# Fallback to S if model name not found
config = progressive_configs.get(MODEL_NAME, progressive_configs["EfficientNetV2-S"])

IMG_SIZE_START, IMG_SIZE_END = config["img_size"]
RANDAUG_MAG_START, RANDAUG_MAG_END = config["randaug_mag"]
MIXUP_ALPHA_START, MIXUP_ALPHA_END = config["mixup_alpha"]
DROPOUT_START, DROPOUT_END = config["dropout"]

class ProgressiveScheduler:
    def __init__(self, total_epochs, stages=4):
        self.total_epochs = total_epochs
        self.stages = stages
        self.epochs_per_stage = math.ceil(total_epochs / stages)
    
    def step(self, epoch):
        stage = min(epoch // self.epochs_per_stage, self.stage - 1)
        progress = stage / max(1, self.stages - 1)
        
        img_size = int(IMG_SIZE_START + progress * (IMG_SIZE_END - IMG_SIZE_START))
        randaug = int(
            RANDAUG_MAG_START + progress * (RANDAUG_MAG_END - RANDAUG_MAG_START)
        )
        mixup = MIXUP_ALPHA_START + progress * (MIXUP_ALPHA_END - MIXUP_ALPHA_START)
        dropout = DROPOUT_START + progress * (DROPOUT_END - DROPOUT_START)

        return img_size, randaug, mixup, dropout

# 3. Training Data Preparation

In [ ]:
stats = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def get_transforms(img_size, randaug_mag):
    train_transform = transforms.Compose([
        transforms.Resize(img_size + 32),
        transforms.RandomRotation(15),
        transforms.RandomCrop(img_size),
        transforms.RandAugment(num_ops=2, magnitude=randaug_mag),
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])

    test_transform = transforms.Compose([
        transforms.Resize(img_size + 32),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])
    
    return train_transform, test_transform

In [ ]:
import numpy as np
import random
import os

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_seed(SEED)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
# Download training data as dummy data without transforms
dummy_data = datasets.Food101(root=DATA_PATH, split="train", download=True)

# Split the dummy data into two temp subset
train_size = int(0.8 * len(dummy_data))
val_size = len(dummy_data) - train_size
split_generator = torch.Generator().manual_seed(SEED)

train_tmp_subset, val_tmp_subset = random_split(
    dummy_data, [train_size, val_size], generator=split_generator
)

# Extract the indices and create the subsets with correct transforms
train_indices = train_tmp_subset.indices
val_indices = val_tmp_subset.indices

# 4. Model Architecture

In [22]:
def _make_divisible(old_ch, divisor=8, min_ch=None):
    if min_ch is None:
        min_ch = divisor

    # Calculate new channels
    new_ch = max(min_ch, int(old_ch + divisor // 2) // divisor * divisor)

    # Make sure the new channels doesnt drop too much
    if new_ch < old_ch * 0.9:
        new_ch += divisor
    return new_ch


class ConvNBAct(nn.Sequential):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        groups=1,
        activation=nn.SiLU,
    ):
        padding = (kernel_size - 1) // 2
        activation = (
            nn.Identity() if activation == nn.Identity else nn.SiLU(inplace=True)
        )
        super().__init__(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            activation,
        )


class DropPath(nn.Module):
    def __init__(self, survival_prob: float = 0.8):
        super().__init__()
        self.survival_prob = survival_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.survival_prob == 1.0:
            return x

        # Create a empty tensor
        batch_size = x.shape[0]
        noise = torch.empty(batch_size, 1, 1, 1, device=x.device)

        # Fill the tensor with 1s and 0s
        noise.bernoulli_(self.survival_prob)

        # Scale up the signal to make sure that the total training signal
        # is the same with eval, otherwise the prediction will be ruined.
        if self.survival_prob > 0:
            noise.div_(self.survival_prob)

        return x * noise


class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels, hidden_channels, squeeze_factor=4):
        super().__init__()
        squeeze_channels = _make_divisible(in_channels // 4)
        self.block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(hidden_channels, squeeze_channels, kernel_size=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(squeeze_channels, hidden_channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.block(x)


class MBConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride,
        expand_ratio,
        fused=False,
        survival_prob=0.8,
    ):
        super().__init__()
        hidden_channels = _make_divisible(in_channels * expand_ratio)
        self.use_res_connect = stride == 1 and in_channels == out_channels
        self.drop_path = (
            DropPath(survival_prob) if self.use_res_connect else nn.Identity()
        )

        layers = []

        # Regular 3x3 conv for FusedMBConv
        if fused:
            layers.append(
                ConvNBAct(
                    in_channels=in_channels,
                    out_channels=hidden_channels,
                    kernel_size=3,
                    stride=stride,
                    groups=1,
                    activation=nn.SiLU,
                )
            )
        else:
            layers.extend([
                    # 1x1 expand conv
                    ConvNBAct(
                        in_channels=in_channels,
                        out_channels=hidden_channels,
                        kernel_size=1,
                        stride=1,
                        groups=1,
                        activation=nn.SiLU,
                    ),
                    # 3x3 dw conv
                    ConvNBAct(
                        in_channels=hidden_channels,
                        out_channels=hidden_channels,
                        kernel_size=3,
                        stride=stride,
                        groups=hidden_channels,
                        activation=nn.SiLU,
                    ),
            ])

        # SqueezeExcitation
        if not fused:
            layers.append(SqueezeExcitation(in_channels, hidden_channels))

        # 1x1 projection conv
        layers.append(
            ConvNBAct(
                in_channels=hidden_channels,
                out_channels=out_channels,
                kernel_size=1,
                activation=nn.Identity,
            )
        )

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        out = self.block(x)
        if self.use_res_connect:
            out = self.drop_path(out)
            out = x + out
        return out


class EfficientNetV2(nn.Module):
    def __init__(
        self, model_name, in_channels, num_classes, dropout, drop_connect_rate=0.2
    ):
        super().__init__()

        # [Fused, exp, stride, out, num_layers]
        model_cfg = {
            "EfficientNetV2-S": {
                "configs": [
                    [True, 1, 1, 24, 2],
                    [True, 4, 2, 48, 4],
                    [True, 4, 2, 64, 4],
                    [False, 4, 2, 128, 6],
                    [False, 6, 1, 160, 9],
                    [False, 6, 2, 256, 15],
                ],
                "first_out": 24,
            },
            "EfficientNetV2-M": {
                "configs": [
                    [True, 1, 1, 24, 3],
                    [True, 4, 2, 48, 5],
                    [True, 4, 2, 80, 5],
                    [False, 4, 2, 160, 7],
                    [False, 6, 1, 176, 14],
                    [False, 6, 2, 304, 18],
                    [False, 6, 1, 512, 5],
                ],
                "first_out": 24,
            },
            "EfficientNetV2-L": {
                "configs": [
                    [True, 1, 1, 32, 4],
                    [True, 4, 2, 64, 7],
                    [True, 4, 2, 96, 7],
                    [False, 4, 2, 192, 10],
                    [False, 6, 1, 224, 19],
                    [False, 6, 2, 384, 25],
                    [False, 6, 1, 640, 7],
                ],
                "first_out": 32,
            },
            "EfficientNetV2-XL": {
                "configs": [
                    [True, 1, 1, 32, 4],
                    [True, 4, 2, 64, 8],
                    [True, 4, 2, 96, 8],
                    [False, 4, 2, 192, 16],
                    [False, 6, 1, 256, 24],
                    [False, 6, 2, 512, 32],
                    [False, 6, 1, 640, 8],
                ],
                "first_out": 32,
            },
        }

        total_blocks = sum(
            math.ceil(num_layers)
            for _, _, _, _, num_layers in model_cfg[model_name]["configs"]
        )
        block_id = 0

        # The first 3x3 conv
        layers = []
        out_channels = model_cfg[model_name]["first_out"]
        layers.append(
            ConvNBAct(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=2,
            )
        )
        in_channels = out_channels

        #
        for fused, exp, stride, out, num_layers in model_cfg[model_name]["configs"]:
            for i in range(num_layers):
                stride = stride if i == 0 else 1
                survival_prob = 1.0 - drop_connect_rate * float(block_id) / total_blocks

                layers.append(
                    MBConv(
                        in_channels=in_channels,
                        out_channels=out,
                        stride=stride,
                        expand_ratio=exp,
                        fused=fused,
                        survival_prob=survival_prob,
                    )
                )
                in_channels = out
                block_id += 1

        layers.extend([
            ConvNBAct(
                in_channels=in_channels,
                out_channels=1280,
                kernel_size=1,
            ),
            nn.AdaptiveAvgPool2d(1),
        ])

        self.block = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1280, num_classes),
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.block(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [27]:
print(f"Using: {MODEL_NAME}")
model = EfficientNetV2(
    MODEL_NAME,
    IN_CHANNELS,
    NUM_CLASSES,
    DROPOUT,
).to(device)

print(f"Total parameters: {(sum(p.numel() for p in model.parameters())/1e6):.2f}M")

Using: EfficientNetV2-S
Total parameters: 20.31M


# 5. Training Preparation

In [ ]:
class EarlyStopping:
    def __init__(
        self, patience=10, delta=0, verbose=False, save_path="best_checkpoint.pth"
    ):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.save_path = save_path
        self.verbose = verbose

        self.early_stop = False
        self.counter = 0
        self.best_loss = None
    
    def __call__(self, model, val_loss):
        # 1. For the first epoch
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        
        # 2. If the loss didnt decrease as expect
        elif val_loss >= self.best_loss - self.delta:
            self.counter += 1
            print(f"Early Stopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        
        # 3. The loss decreased properly
        else:
            self.counter = 0
            self.best_loss = val_loss
            self.save_checkpoint(model)

    def save_checkpoint(self, model):
        if self.verbose:
            print("Saving best checkpoint ...")
        state_dict = (
            model.module.state_dict()
            if hasattr(model, "module")
            else model.state_dict()
        )
        torch.save(state_dict, self.save_path)

# 6. Train

# 7. GradCAM